In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("Spotify_Youtube.csv")
print(df.shape)
df.info()
df.head()

(20718, 28)
<class 'pandas.DataFrame'>
RangeIndex: 20718 entries, 0 to 20717
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        20718 non-null  int64  
 1   Artist            20718 non-null  str    
 2   Url_spotify       20718 non-null  str    
 3   Track             20718 non-null  str    
 4   Album             20718 non-null  str    
 5   Album_type        20718 non-null  str    
 6   Uri               20718 non-null  str    
 7   Danceability      20716 non-null  float64
 8   Energy            20716 non-null  float64
 9   Key               20716 non-null  float64
 10  Loudness          20716 non-null  float64
 11  Speechiness       20716 non-null  float64
 12  Acousticness      20716 non-null  float64
 13  Instrumentalness  20716 non-null  float64
 14  Liveness          20716 non-null  float64
 15  Valence           20716 non-null  float64
 16  Tempo             20716 non-null  float

,Unnamed: 0,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,...,Url_youtube,Title,Channel,Views,Likes,Comments,Description,Licensed,official_video,Stream
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


In [3]:
print(df.isnull().sum().sort_values(ascending=False))
print(df.duplicated(subset=['Artist', 'Track']).sum())
print((df['Key'] == -1).sum())
print(df['Stream'].describe())
print(df['Album_type'].value_counts())

Description         876
Stream              576
Comments            569
Likes               541
Channel             470
Views               470
Title               470
Url_youtube         470
official_video      470
Licensed            470
Energy                2
Key                   2
Danceability          2
Loudness              2
Tempo                 2
Duration_ms           2
Instrumentalness      2
Acousticness          2
Liveness              2
Speechiness           2
Valence               2
Album_type            0
Uri                   0
Album                 0
Unnamed: 0            0
Artist                0
Url_spotify           0
Track                 0
dtype: int64
82
0
count    2.014200e+04
mean     1.359422e+08
std      2.441321e+08
min      6.574000e+03
25%      1.767486e+07
50%      4.968298e+07
75%      1.383581e+08
max      3.386520e+09
Name: Stream, dtype: float64
Album_type
album          14926
single          5004
compilation      788
Name: count, dtype: int64


In [4]:
drop_cols = ['Url_spotify', 'Uri', 'Url_youtube', 'Description', 'Title',
             'Artist', 'Track', 'Album', 'Channel',
             'Views', 'Likes', 'Comments', 'Licensed', 'official_video']

df_clean = df.drop(columns=drop_cols)

df_clean['Key'] = df_clean['Key'].fillna(-1).astype(int).astype(str)

feature_cols = ['Danceability', 'Energy', 'Key', 'Loudness', 'Speechiness',
                 'Acousticness', 'Instrumentalness', 'Liveness', 'Valence',
                 'Tempo', 'Duration_ms', 'Album_type']
target_col = 'Stream'

df_clean = df_clean.dropna(subset=feature_cols + [target_col])
df_clean = df_clean[df_clean[target_col] > 0]
df_clean['log_stream'] = np.log1p(df_clean[target_col])

numeric_feats = ['Danceability', 'Energy', 'Loudness', 'Speechiness',
                  'Acousticness', 'Instrumentalness', 'Liveness', 'Valence',
                  'Tempo', 'Duration_ms']
for col in numeric_feats:
    assert pd.api.types.is_numeric_dtype(df_clean[col]), f"{col} is not numeric"

assert df_clean[feature_cols].isnull().sum().sum() == 0
assert (df_clean[target_col] > 0).all()
print("Cleaning checks passed:", df_clean.shape)

Cleaning checks passed: (20140, 15)


In [5]:
X = df_clean[feature_cols]
y = df_clean['log_stream']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)